# Linux 101 with $\color{red}{PWN.COLLEGE}$


# Pondering Paths

### Program and absolute paths
No Linux, todo arquivo e programa vive dentro de uma árvore de diretórios que começa na raiz, representada por uma barra (/). <br>
- Caminho Absoluto (Absolute Path): É o endereço completo de um arquivo ou programa, começando obrigatoriamente pela raiz (/). Não importa em qual diretório você esteja no momento (current working directory), o caminho absoluto sempre funcionará igual.

### Postion thy self 
- Todo processo precisa estar "em algum lugar" da árvore de diretórios
- O comando pwd mostra exatamente qual é esse lugar
- O comando cd (Change Directory) altera a sua posição atual.
- O caractere ~ (til) no prompt (hacker@dojo:~$) é um atalho para o seu diretório home (/home/hacker).

###  Position elsewhere 
Muitos programas no pwn.college (e no Linux em geral) verificam o diretório de trabalho atual (CWD) antes de fazer qualquer coisa. Eles fazem isso por vários motivos:
- Para salvar arquivos de log ou temporários no lugar correto.
- Por questões de segurança ou lógica interna do binário.
- Para dificultar a execução direta se você não estiver na pasta designada.

### Implicit relative paths, from /
É um endereço baseado na sua posição atual. Por exemplo, ./run significa "execute o arquivo run que está no diretório atual" (./). <br>
A variável de ambiente PATH:
Quando você digita apenas o nome de um comando (ex: run em vez de ./run), o shell não sai procurando no diretório atual por motivos de segurança. Ele busca em uma lista estrita de diretórios definidos na variável PATH (como /usr/bin, /bin, etc.). O diretório atual (.) não está no PATH por padrão. <br>
O problema do desafio (from /):
Neste nível, você precisa rodar o programa run (que está em /challenge/run), mas a pegadinha é que você precisa estar posicionado na raiz (/) e tentar executá-lo usando um caminho relativo implícito ou direto.

### Explicit relative paths, from /
Caminho Relativo Explícito: Significa usar obrigatoriamente os símbolos que definem o diretório atual (.) ou o diretório pai (..) para indicar onde o arquivo está. <br>
O ponto (.) representa exatamente o diretório em que você está posicionado agora. Portanto, ./ diz explicitamente ao shell: "procure o arquivo dentro desta pasta atual e em nenhum outro lugar".

### Implicit relative path
Caminho Relativo Implícito: É quando você referencia um arquivo ou diretório a partir da sua posição atual sem colocar o ./ na frente.<br>
Por exemplo, se você está dentro da pasta /challenge, em vez de rodar ./run, você roda apenas run. O shell entende implicitamente que você quer o arquivo do diretório atual (se configurado ou dependendo de como o binário/desafio intercepta a execução).

### Home sweet home
O Diretório Home (\~): É a sua "casa" no sistema operacional (geralmente /home/hacker nos dojos do pwn.college). É onde você tem permissões totais para criar, modificar e deletar arquivos livremente. <br>
O Til (\~): Funciona como um atalho universal para o seu diretório home, não importa onde você esteja na árvore de diretórios.
- Se você está em /usr/bin e digita cd ~, você vai direto para /home/hacker.
- O comando cd sozinho (sem nenhum argumento) também te joga direto para o home.



# Comprehending Commands

---
## Leitura e Filtragem
### cat (Ler arquivos):
cat /flag

### grep (Buscar padrões em arquivos):
grep "pwn.college" /challenge/data.txt

---
## Comparação
### diff (Comparar arquivos linha por linha):
diff arquivo1.txt arquivo2.txt

---
## Gestão de Arquivos
### touch (Criar arquivo vazio):
touch /tmp/meu_arquivo.txt

### rm (Remover arquivo):
rm /tmp/meu_arquivo.txt

### cp (Copiar arquivo):
cp /flag /tmp/copia_da_flag

### mv (Mover ou renomear arquivo):
mv /tmp/copia_da_flag /home/hacker/flag_segura

### mkdir (Criar diretório):
mkdir /tmp/meu_diretorio

---
## Localização e Listagem
### ls -a (Listar arquivos, incluindo ocultos):
ls -a /

### find (Encontrar arquivos no sistema):
find / -name "flag"

---
## Links
### ln -s (Criar link simbólico / atalho):
ln -s /flag /home/hacker/meu_link

---

# File Globbing

---
## Matching with * 
O * funciona como um "coringa" que representa qualquer sequência de caracteres (de zero a infinitos).
- *.txt -> Seleciona tudo que termina com .txt
- data* -> Seleciona tudo que começa com data (ex: data1, database, data_final.txt).
- * -> selecionatudo absolutamente no diretório atual

---
## Matching with ? 
Enquanto o * representa qualquer quantidade de caracteres (de zero a infinitos), o ? representa exatamente um único caractere.
- Se você tem arquivos chamados file1, file2 e file10:
  - file? vai casar com file1 e file2, mas não vai casar com file10 (porque o 10 tem dois caracteres, e o ? exige estritamente um).
  - file?? casaria com file10.

---
## Matching with []
Os colchetes permitem que você defina um conjunto de caracteres permitidos para uma única posição no nome do arquivo.
- file_[abc] -> Vai procurar arquivos que comecem com file_ e terminem exatamente com a, b ou c (ex: file_a, file_b, file_c). Se houver um file_d, ele não entra.
- Intervalos (Ranges): Você também pode usar um hífen dentro dos colchetes para definir um intervalo.
  - file_[a-z] -> Qualquer letra minúscula de a até z.
  - file_[0-9] -> Qualquer dígito de 0 até 9.

---
## Multiple Globs
O shell não se limita a apenas um curinga por comando. Você pode combinar quantos globs quiser no mesmo caminho para filtrar estruturas inteiras de diretórios ou nomes de arquivos complexos.
- Por exemplo: /challenge/files/[abc]*_??.txt
  - [abc] -> O nome precisa começar com a, b ou c.
  - * -> Seguido de qualquer quantidade de caracteres no meio.
  - _?? -> Seguido de um sublinhado e exatamente dois caracteres antes da extensão.
  - *.txt -> Terminando com .txt.

---
## Exclusionary Globbing
Dentro dos colchetes [], se você colocar um ponto de interrogação ! (ou em alguns shells o acento circunflexo ^) logo após abrir o colchete, ele inverte a lógica: ele passa a significar "qualquer caractere EXCETO estes".
- file_[!a] -> Vai casar com file_b, file_c, file_z, mas vai ignorar file_a.
- file_[!0-9] -> Vai casar com qualquer arquivo que termine com uma letra ou símbolo, mas exclui os que terminam com números de 0 a 9.

---
## Tab Completion
Ninguém na vida real digita caminhos gigantescos ou nomes de arquiv
os complexos letra por letra. Nós usamos a tecla Tab.
Quando você começa a digitar um comando, o nome de um arquivo ou um caminho e aperta a tecla Tab:
- Autocompletação: O shell tenta adivinhar o resto do nome sozinho. Se houver apenas uma opção óbvia, ele preenche o resto do caminho instantaneamente.
- Listagem rápida: Se houver várias opções que começam do mesmo jeito (por exemplo, file_apple e file_banana), se você apertar Tab duas vezes, o shell te mostra todas as opções disponíveis na tela para você decidir.

---

# Practing Piping

---
## Redirecting Output
O símbolo > é como um "funil" que pega o que seria impresso no terminal e joga para dentro de um arquivo.
- Sintaxe: comando > arquivo
  - O Linux executa o comando
  - O resultado que iria para a tela é redirecionado
  - Se o arquivo não existir, ele é criado. Se já existir, ele é sobrescrito (o conteúdo antigo é apagao)

---
## Appending Output
Depois de aprender a sobrescrever arquivos com o >, agora você vai aprender a acrescentar dados ao final de um arquivo sem apagar o que já está lá.
- \> (Sobrescreve): Se o arquivo já tiver conteúdo, ele limpa tudo e escreve o novo resultado por cima.
- \>> (Anexa / Append): Se o arquivo já existir, ele mantém o conteúdo antigo e joga o novo resultado lá no finalzinho do arquivo. Se o arquivo não existir, ele cria um novo (assim como o \>).

---
## Redirecting Errors
Por padrão, o Linux lida com os fluxos de dados usando números chamados de file descriptors:
- 0 -> stdin (Entrada padrão / teclado)
- 1 -> stdout (Saída padrão / o sucesso)
- 2 -> stderr (Saída de erro / falhas)

---
## Redirecting Input
Normalmente, quando um programa interativo ou de leitura espera que você digite algo, ele fica aguardando o seu input no terminal (o stdin ou descritor 0).
- comando < arquivo_de_entrada.txt

---
## Grepping Stored Result
Muitas vezes, um comando gera uma quantidade massiva de dados (milhares de linhas). Se você tentar pipear direto (| grep), por algum motivo o desafio pode exigir que você salve o resultado primeiro, ou talvez você queira analisar os dados com calma depois.
- /challenge/run > /tmp/saida_gigante.txt
- grep "pwn.college" /tmp/saida_gigante.txt

---
## Grepping Live Output
Em vez de rodar o comando, salvar em um arquivo gigante com >, e depois rodar o grep no arquivo, você junta os dois processos instantaneamente usando o |:
- /challenge/run | grep "pwn.college"

---
## Grepping Errors
Por padrão, o operador de pipe (|) só consegue enxergar a saída padrão (stdout / canal 1). Ele ignora completamente o canal de erro (stderr / canal 2).
- /challenge/run | grep "pwn"

---
## Filtering with Grepping -V
A flag -v (de invert-match ou inversão) instrui o grep a fazer exatamente o contrário do seu comportamento padrão:
- Comportamento Normal (grep "palavra"): Mostra apenas as linhas que contêm "palavra".
- Comportamento Invertido (grep -v "palavra"): Descarta todas as linhas que contêm "palavra" e mostra todas as outras que sobrarem.

---
## Filtering with Sed
Enquanto o grep é focado em encontrar e mostrar linhas, o sed é focado em transformar o texto em tempo real conforme ele passa pelo pipe.
- comando | sed 's/antigo/novo/'

---
## Duplicating Piped Data with Tee
Imagine um encanamento de água que chega a um ponto onde você quer colocar uma derivação (um "T"): uma parte da água continua indo para o cano principal, e a outra parte é desviada para um balde.
- Ele pega o dado que está vindo pelo pipe.
- Ele grava esse dado em um arquivo (o seu balde).
- Ele passa adiante exatamente o mesmo dado intacto para o próximo comando no pipe (o cano principal).
- comando1 | tee arquivo_intermediario.txt | comando2

---
## Process Substitution for Input
Normalmente, o operador < espera um arquivo real no disco:
- Normalmente, o operador < espera um arquivo real no disco:
- comando < arquivo.txt
- Mas e se você quiser que a entrada venha da saída de outro comando (por exemplo, gerando dados na hora, filtrando algo ou combinando saídas), sem precisar criar um arquivo temporário no disco (/tmp)?
- comando <(outro_comando)

---
## Wrinting to Multiple Programs
Enquanto o <(...) cria um descritor para ser lido como arquivo de entrada, a sintaxe >(...) faz o oposto: ela cria um descritor para receber dados de escrita e passá-los como se fossem arquivos de saída para o comando que está sendo executado lá dentro.

Se você quiser enviar a mesma saída para dois comandos diferentes ao mesmo tempo, você pode usar o operador de redirecionamento combinado com a substituição de processo:
- comando_gerador > >(programa1) > >(programa2)

---
## Split-Piping stderr and stdout
Como vimos antes, o pipe normal (|) captura apenas o canal 1 (stdout). Se você usar 2>&1, você junta tudo em um canal só.
Mas e se você precisar que o sucesso vá para um comando (ou arquivo) e o erro vá para outro lugar completamente diferente, sem misturar os dois?

Para fazer essa divisão cirúrgica, precisamos usar recursos avançados de redirecionamento de descritores de arquivos do Bash.
- { /challenge/run 2>&3 | processar_erros; } 3>&1 | processar_sucesso

---

## Named Pipes
Um Named Pipe (também conhecido como FIFO - First In, First Out) é um arquivo especial no sistema de arquivos que atua como um canal de comunicação.

A grande sacada é que ele parece e se comporta como um arquivo comum, mas por trás dos panos ele funciona como um cano de dados:
- Persistência: Ele tem um nome real e fica visível no diretório (você o vê com ls -l, onde a permissão começa com a letra p).
- Sincronização de Fluxo: Quando um processo tenta escrever nele, ele trava (fica bloqueado) até que outro processo venha e leia os dados (e vice-versa). É como um ponto de encontro físico para os dados passarem de um programa para o outro.
- mkfifo meu_pipe
- echo "dados secretos" > meu_pipe
- cat < meu_pipe

---

# Shell Variables

## Printing Variables
Como vimos, o shell trata o nome da variável como um identificador. Para dizer ao shell: "Ei, substitua este nome pelo valor que está guardado nele", usamos o prefixo \\$.
O echo é o comando clássico para imprimir coisas no terminal. Quando você combina ele com o \$, ele se torna a ferramenta principal para ver o que tem dentro da variável:
- Comando: echo \$FLAG

---
## Setting Variables
No Bash, existe uma diferença fundamental entre uma variável local e uma variável de ambiente. Entender isso vai te salvar de muitos problemas de "comando não encontrado" ou "variável vazia".
- Variável Local (Apenas para o Shell atual):
  - Quando você faz NOME=valor, ela existe apenas no seu shell atual. Se você rodar um programa (como o /challenge/run), esse programa não saberá que essa variável existe, porque ela é privada.
- Variável de Ambiente (Passada para subprocessos)
  - Para que um programa que você executa (o /challenge/run) consiga "ler" a variável, você precisa exportá-la.
- export NOME=hacker
- echo $NOME

---
## Multi-Word Variables
Por padrão, o shell usa o espaço como um delimitador (um separador de argumentos).

Se você tentar fazer isto:
- MENSAGEM=Ola mundo
O shell vai achar que você está tentando definir a variável MENSAGEM="Ola" e, logo em seguida, tentar executar o comando mundo. Vai dar erro de comando não encontrado.

A Solução: Usando Aspas Duplas ("...")
Para agrupar várias palavras em um único valor de variável, você deve usar aspas:

- MENSAGEM="Ola mundo de sistemas"

---
## Exporting Variables
Quando você abre o terminal, o shell cria um processo. Qualquer variável que você define sem o comando export fica confinada à memória daquele shell específico. Ela é uma variável local.
- MINHA_VAR="secreto"
- export MINHA_VAR

---
## Printing Exported Variables
Quando você usa echo \$VAR, você está imprimindo uma variável que pode ser puramente local (criada apenas como VAR=123) ou exportada. Mas e quando você quer ver tudo o que está disponível no seu ambiente de execução?
- echo $VAR: Ideal para checar o valor de uma variável específica que você já sabe o nome.
- printenv ou env: Lista todas as variáveis de ambiente que foram exportadas e estão ativas na sua sessão atual.

---
## Storing Command Output
Se você tentar fazer isso:
- MINHA_VAR=ls
Você só guardou a palavra ls

Para guardar o comando ls:
- MINHA_VAR=$(ls)

---
## Reading Input
O comando read é a principal ferramenta do Bash para capturar a entrada padrão (stdin) — ou seja, o que é digitado no terminal — e armazenar diretamente dentro de uma variável.
- read MINHA_VARIAVEL
  - O terminal pausa a execução e fica piscando, esperando você digitar alguma coisa.
  - Você digita um texto (ex: senha_secreta) e aperta Enter.
  - O Bash pega o que você digitou e guarda instantaneamente dentro da variável MINHA_VARIAVEL.

- read -p "Digite seu nome: " NOME
- echo "Olá, \$NOME!"
  - -s (Silent): Oculta o que você está digitando na tela (essencial para senhas ou dados sensíveis).

 ---
## Reading Files
No Linux, tudo é arquivo — e ler arquivos de texto é uma tarefa diária para qualquer desenvolvedor ou analista de segurança. Dependendo do tamanho do arquivo e do que você precisa fazer, existem diferentes ferramentas para essa missão.
- cat nome_do_arquivo.txt
- head -n 5 arquivo.txt  # Mostra as 5 primeiras linhas
- tail -n 20 arquivo.txt # Mostra as últimas 20 linhas

---